# Generación de Imágenes con IA

> Aprende a generar imágenes con DALL-E 3 desde texto, comparar prompts básicos vs detallados,
> y combinar Claude + DALL-E en un pipeline creativo.

## Objetivos
- Generar imágenes con DALL-E 3 desde texto
- Guardar y mostrar imágenes en el notebook con `IPython.display`
- Comparar el impacto de prompts básicos vs detallados
- Construir un pipeline Claude → DALL-E (Claude mejora el prompt, DALL-E genera)

## 1. Instalación de dependencias

In [ ]:
%pip install openai anthropic pillow requests --quiet

## 2. Configuración inicial

In [ ]:
import openai
import anthropic
import requests
import os
from pathlib import Path
from IPython.display import Image, display, HTML
from datetime import datetime

# Clientes
openai_client = openai.OpenAI()   # usa OPENAI_API_KEY
claude_client = anthropic.Anthropic()  # usa ANTHROPIC_API_KEY

MODELO_CLAUDE = "claude-haiku-4-5-20251001"
MODELO_DALLE = "dall-e-3"

# Directorio para guardar imágenes
DIR_IMAGENES = Path("imagenes_generadas")
DIR_IMAGENES.mkdir(exist_ok=True)

print(f"Clientes inicializados.")
print(f"Generador de imágenes: {MODELO_DALLE}")
print(f"Mejorador de prompts: {MODELO_CLAUDE}")
print(f"Imágenes se guardarán en: {DIR_IMAGENES.absolute()}")

## 3. Generar imagen con DALL-E 3 desde texto

In [ ]:
def generar_imagen(prompt: str, nombre_archivo: str = None,
                   tamano: str = "1024x1024", calidad: str = "standard") -> dict:
    """Genera una imagen con DALL-E 3 y la guarda localmente.

    Args:
        prompt: Descripción de la imagen a generar
        nombre_archivo: Nombre del archivo (sin extensión). Autogenerado si None.
        tamano: Resolución (1024x1024, 1024x1792, 1792x1024)
        calidad: 'standard' o 'hd'

    Returns:
        Dict con url, ruta_local, prompt_revisado
    """
    print(f"Generando imagen...")
    print(f"Prompt: '{prompt[:80]}{'...' if len(prompt) > 80 else ''}'")

    respuesta = openai_client.images.generate(
        model=MODELO_DALLE,
        prompt=prompt,
        size=tamano,
        quality=calidad,
        n=1
    )

    url = respuesta.data[0].url
    prompt_revisado = respuesta.data[0].revised_prompt or prompt

    # Descargar y guardar localmente
    if nombre_archivo is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        nombre_archivo = f"imagen_{timestamp}"

    ruta = DIR_IMAGENES / f"{nombre_archivo}.png"
    datos_imagen = requests.get(url, timeout=30).content
    ruta.write_bytes(datos_imagen)

    print(f"Imagen guardada en: {ruta}")
    return {
        "url": url,
        "ruta_local": str(ruta),
        "prompt_original": prompt,
        "prompt_revisado": prompt_revisado
    }

# Generar primera imagen
PROMPT_BASICO = "Un robot leyendo un libro"
resultado1 = generar_imagen(PROMPT_BASICO, "robot_basico")
print(f"\nPrompt revisado por DALL-E: {resultado1['prompt_revisado'][:120]}...")
display(Image(filename=resultado1["ruta_local"], width=400))

## 4. Comparar prompts básico vs detallado

Un prompt más específico produce resultados mucho mejores con DALL-E 3.

In [ ]:
# Prompt detallado para el mismo concepto
PROMPT_DETALLADO = """Un robot humanoide de acero pulido con detalles cromados,
sentado en una biblioteca antigua con estantes de madera llenos de libros,
sosteniendo un libro abierto con sus manos mecánicas articuladas,
iluminación cálida de lámpara de escritorio, estilo ilustración digital detallada,
atmósfera tranquila y contemplativa, alta calidad, 8K"""

resultado2 = generar_imagen(PROMPT_DETALLADO, "robot_detallado")
display(Image(filename=resultado2["ruta_local"], width=400))

print("\n=== COMPARACIÓN DE PROMPTS ===")
print(f"\nPrompt BÁSICO ({len(PROMPT_BASICO)} caracteres):")
print(f"  '{PROMPT_BASICO}'")
print(f"\nPrompt DETALLADO ({len(PROMPT_DETALLADO)} caracteres):")
print(f"  '{PROMPT_DETALLADO[:120]}...'")
print("\nLecciones aprendidas:")
print("  - Especificar el estilo visual (digital, fotorrealista, pintura al óleo)")
print("  - Describir el entorno y la iluminación")
print("  - Incluir detalles de materiales y texturas")
print("  - Definir la atmósfera o emoción deseada")

## 5. Pipeline Claude → DALL-E

Claude mejora el prompt del usuario y luego DALL-E genera la imagen. Combina lo mejor de ambos modelos.

In [ ]:
def mejorar_prompt_con_claude(idea_basica: str) -> str:
    """Usa Claude para convertir una idea simple en un prompt detallado para DALL-E."""
    instruccion = f"""Convierte esta idea básica en un prompt detallado y específico para DALL-E 3.
El prompt debe incluir: estilo visual, iluminación, materiales, atmósfera, composición y calidad.
Escríbelo en inglés (DALL-E funciona mejor con prompts en inglés).
Máximo 200 palabras. Responde ÚNICAMENTE con el prompt mejorado, sin explicaciones.

Idea básica: {idea_basica}"""

    respuesta = claude_client.messages.create(
        model=MODELO_CLAUDE,
        max_tokens=300,
        messages=[{"role": "user", "content": instruccion}]
    )
    return respuesta.content[0].text.strip()


def pipeline_claude_dalle(idea: str, nombre: str = None) -> dict:
    """Pipeline completo: Claude mejora el prompt y DALL-E genera la imagen."""
    print(f"PASO 1: Claude mejora el prompt...")
    prompt_mejorado = mejorar_prompt_con_claude(idea)
    print(f"Prompt mejorado: {prompt_mejorado[:150]}...\n")

    print(f"PASO 2: DALL-E genera la imagen...")
    resultado = generar_imagen(prompt_mejorado, nombre or "pipeline_resultado")
    resultado["idea_original"] = idea
    resultado["prompt_claude"] = prompt_mejorado
    return resultado


# Ejecutar el pipeline
IDEA_USUARIO = "una ciudad del futuro con energía solar"

print("=" * 60)
print(f"PIPELINE CLAUDE → DALL-E")
print(f"Idea del usuario: '{IDEA_USUARIO}'")
print("=" * 60)

resultado_pipeline = pipeline_claude_dalle(IDEA_USUARIO, "ciudad_futuro_pipeline")

print("\nResultado final:")
display(Image(filename=resultado_pipeline["ruta_local"], width=500))

print("\nResumen del pipeline:")
print(f"  Idea original: '{resultado_pipeline['idea_original']}'")
print(f"  Prompt Claude: '{resultado_pipeline['prompt_claude'][:100]}...'")
print(f"  Archivo: {resultado_pipeline['ruta_local']}")